In [1]:
import os
import cv2
import numpy as np
import time as time_module
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

In [3]:
# =============================
# CONFIG
# =============================
IMAGE_DIR = "images"
OUTPUT_DIR = "output_sam1"

SAM_CHECKPOINT = "sam_vit_h_4b8939.pth"
MODEL_TYPE = "vit_h"
DEVICE = "cuda"   # change to "cuda" if available

# Object-level tuning

# fewer objects	Increase MIN_MASK_AREA
# More objects	Decrease MIN_MASK_AREA
# Aggressive merge	Lower OVERLAP_IOU_THRESH
# Faster	POINTS_PER_SIDE = 8
# Cleaner	Increase STABILITY_THRESH
POINTS_PER_SIDE = 8
MIN_MASK_AREA = 20000
PRED_IOU_THRESH = 0.9
STABILITY_THRESH = 0.97
OVERLAP_IOU_THRESH = 0.10

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================
# Load SAM
# =============================
sam = sam_model_registry[MODEL_TYPE](checkpoint=SAM_CHECKPOINT)
sam.to(device=DEVICE)

mask_generator = SamAutomaticMaskGenerator(
    sam,
    points_per_side=POINTS_PER_SIDE,
    pred_iou_thresh=PRED_IOU_THRESH,
    stability_score_thresh=STABILITY_THRESH,
    min_mask_region_area=MIN_MASK_AREA,
)

# =============================
# IoU function
# =============================
def mask_iou(mask1, mask2):
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    return intersection / union if union > 0 else 0

C:\Users\USERAS\AppData\Roaming\Python\Python311\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


In [4]:
# =============================
# Process images
# =============================
start_time = 0
end_time = 0
time = 0
for image_file in os.listdir(IMAGE_DIR):
    start_time = time_module.time()
    if not image_file.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    image_path = os.path.join(IMAGE_DIR, image_file)
    image_name = os.path.splitext(image_file)[0]

    print(f"\nProcessing: {image_file}")

    out_dir = os.path.join(OUTPUT_DIR, image_name)
    os.makedirs(out_dir, exist_ok=True)

    # Load image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Generate masks
    masks = mask_generator.generate(image)

    print(f"Initial masks: {len(masks)}")

    # Sort masks by area (largest first)
    masks = sorted(masks, key=lambda x: x["area"], reverse=True)

    # =============================
    # Object-level filtering
    # =============================
    final_masks = []

    for mask_data in masks:
        mask = mask_data["segmentation"]

        keep = True
        for kept in final_masks:
            iou = mask_iou(mask, kept["segmentation"])
            if iou > OVERLAP_IOU_THRESH:
                keep = False
                break

        if keep:
            final_masks.append(mask_data)

    print(f"Object-level masks: {len(final_masks)}")

    # =============================
    # Save cropped segments
    # =============================
    for i, mask_data in enumerate(final_masks):
        mask = mask_data["segmentation"]

        segmented = np.zeros_like(image)
        segmented[mask] = image[mask]

        x, y, w, h = mask_data["bbox"]
        crop = segmented[int(y):int(y+h), int(x):int(x+w)]

        crop_bgr = cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)

        out_path = os.path.join(out_dir, f"object_{i}.png")
        cv2.imwrite(out_path, crop_bgr)

    print(f"Saved to: {out_dir}")
    #execution tyme in real time

    
    end_time = time_module.time()
    time = end_time - start_time

    print(f"time to process: {time}")

print("\n✅ All images processed successfully.")


Processing: bhaat_002.jpg
Initial masks: 6
Object-level masks: 5
Saved to: output_sam1\bhaat_002
time to process: 1.312286615371704

Processing: bhaat_003.jpg
Initial masks: 6
Object-level masks: 6
Saved to: output_sam1\bhaat_003
time to process: 1.1482045650482178

Processing: bhaat_004.jpg
Initial masks: 5
Object-level masks: 2
Saved to: output_sam1\bhaat_004
time to process: 1.46443772315979

Processing: bhaat_005.jpg
Initial masks: 8
Object-level masks: 6
Saved to: output_sam1\bhaat_005
time to process: 1.0657374858856201

Processing: boiled_egg_001.jpg
Initial masks: 19
Object-level masks: 10
Saved to: output_sam1\boiled_egg_001
time to process: 1.3166100978851318

Processing: boiled_egg_002.jpg
Initial masks: 11
Object-level masks: 11
Saved to: output_sam1\boiled_egg_002
time to process: 1.3744096755981445

Processing: boiled_egg_003.jpg
Initial masks: 9
Object-level masks: 7
Saved to: output_sam1\boiled_egg_003
time to process: 1.361466884613037

Processing: boiled_egg_004.jpg
